<a href="https://colab.research.google.com/github/mayagrita/Panoramic_Dent_AI/blob/marla/Panoramic_dent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
import os
import cv2
import numpy as np
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Paths
dataset_path = '/content/drive/MyDrive/dental_dataset/dental_dataset'
# images_folder = os.path.join(dataset_path, "images", "train")
images_folder = os.path.join(dataset_path, "images", "valid")

output_csv = os.path.join(dataset_path, "manual_features_valid2.csv")

# Check if the folder exists
if not os.path.exists(images_folder):
    raise FileNotFoundError(f"Image folder not found: {images_folder}")

# List of image files
image_files = [f for f in os.listdir(images_folder) if f.lower().endswith((".jpg", ".png"))]
print(f"Number of images found: {len(image_files)}")

def extract_manual_features(img_path):
    """
    Extract 5 statistical features from a medical image.
    """
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise ValueError(f"Could not read image: {img_path}")

    features = {}

    # 1. Mean intensity
    features["mean_intensity"] = float(np.mean(img))

    # 2. Standard deviation of intensity
    features["std_intensity"] = float(np.std(img))

    # 3. Number of dark pixels
    dark_pixels = np.sum(img < np.percentile(img, 5))
    features["dark_pixel_count"] = int(dark_pixels)

    # 4. Symmetry score between left and right halves
    left_half = img[:, :img.shape[1] // 2]
    right_half = img[:, img.shape[1] // 2:]
    symmetry_score = abs(np.mean(left_half) - np.mean(right_half))
    features["symmetry_score"] = float(symmetry_score)

    # 5. Image Entropy (randomness in the image)
    hist, _ = np.histogram(img.flatten(), bins=256, range=(0, 256))
    hist = hist / hist.sum()
    entropy = -np.sum(hist * np.log2(hist + 1e-8))
    features["image_entropy"] = float(entropy)

    return features

# Extract features from all images
features_list = []

for i, img_file in enumerate(image_files):
    print(f"[{i+1}/{len(image_files)}] Processing: {img_file}")
    try:
        img_path = os.path.join(images_folder, img_file)
        features = extract_manual_features(img_path)
        features["image_name"] = img_file
        features_list.append(features)
    except Exception as e:
        print(f"Error processing {img_file}: {str(e)}")
        continue

# Convert to DataFrame and save to CSV
df = pd.DataFrame(features_list)
df.to_csv(output_csv, index=False)

print(f"\nFeatures saved to: {output_csv}")
print("First 5 rows:")
print(df.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Number of images found: 2871
[1/2871] Processing: 4066680000-jpg_png_jpg.rf.6451cc4f90887ed6f073de71ff8e3eb1.jpg
[2/2871] Processing: 4066680000-jpg_png_jpg.rf.bb43289ee2e415a43f8ed2c7344be1cd.jpg
[3/2871] Processing: 4066870000-jpg_png_jpg.rf.f4de549d54ac8190167da0ff54317d08.jpg
[4/2871] Processing: 4067600000-jpg_png_jpg.rf.6599fb9629ccb11f404a2800df54cd64.jpg
[5/2871] Processing: 4067600000-jpg_png_jpg.rf.9f46e7fe7e303a54b9014d14da83422b.jpg
[6/2871] Processing: 4068250000-jpg_png_jpg.rf.6212b5f121df626658c0980ce231c225.jpg
[7/2871] Processing: 4068530000-jpg_png_jpg.rf.a96f4733d190df1061e0e1e35a19b4a3.jpg
[8/2871] Processing: 4068540000-jpg_png_jpg.rf.594a9aefa49e696a19238187ea3b4ecf.jpg
[9/2871] Processing: 4068540000-jpg_png_jpg.rf.de1766ccd853f0557fd2d77f14a39362.jpg
[10/2871] Processing: 4069100000-jpg_png_jpg.rf.35bf35ed7c6c16e7df23c79dca03fad5.jpg
[

In [ ]:
# # أولاً: إزالة الإصدارات الحالية
# !pip uninstall -y torch torchvision torchaudio

# # ثم تثبيت الإصدارات المتوافقة (CUDA 11.8 مثلاً)
# !pip install torch==2.0.0 torchvision==0.15.1 torchaudio==2.0.1 --extra-index-url https://download.pytorch.org/whl/cu118

# # إعادة تشغيل النواة (مهم لتفادي الأخطاء)
# import os
# os.kill(os.getpid(), 9)


In [ ]:
# # حذف NumPy الحالي (المتسبب بالمشكلة)
# !pip uninstall -y numpy

# # تثبيت نسخة مستقرة ومتوافقة مع PyTorch و torchvision
# !pip install numpy==1.24.4

# # إعادة تشغيل النواة (مطلوب بعد التثبيت)
# import os
# os.kill(os.getpid(), 9)


In [ ]:
# import os
# os.kill(os.getpid(), 9)


In [ ]:
import torch
from torchvision.models import resnet50, ResNet50_Weights

class CNNFeatureExtractor(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # تحميل ResNet50 بدون الطبقة النهائية
        self.model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.feature_extractor = torch.nn.Sequential(*list(self.model.children())[:-1])

    def forward(self, x):
        with torch.no_grad():
            features = self.feature_extractor(x).flatten(start_dim=1)
        return features

In [ ]:
import os
images_folder = "/content/drive/MyDrive/dental_dataset/dental_dataset/images/train"
output_csv = "/content/drive/MyDrive/dental_dataset/dental_dataset/cnn_features_train.csv"

os.makedirs(os.path.dirname(output_csv), exist_ok=True)

In [ ]:
from torchvision.io import read_image
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ConvertImageDtype(torch.float32),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
cnn_extractor = CNNFeatureExtractor()
cnn_extractor.eval()

image_files = [f for f in os.listdir(images_folder) if f.lower().endswith((".jpg", ".png"))]
features_list = []

for i, img_file in enumerate(image_files):
    print(f"Extracting features from image {i+1}/{len(image_files)}: {img_file}")

    try:
        img_path = os.path.join(images_folder, img_file)
        img = read_image(img_path)
        img = transform(img).unsqueeze(0)  # Shape: [1, 3, 224, 224]

        with torch.no_grad():
            cnn_features = cnn_extractor(img).squeeze(0).cpu().numpy()  # Shape: [2048]

        features_list.append({
            "image_name": img_file,
            **{f"cnn_{j}": cnn_features[j] for j in range(cnn_features.shape[0])}
        })

    except Exception as e:
        print(f"Error processing {img_file}: {str(e)}")
        continue

# Save features to CSV
df_cnn = pd.DataFrame(features_list)
df_cnn.to_csv(output_csv, index=False)

print(f"CNN features saved to: {output_csv}")


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 103MB/s]


Extracting features from image 1/9443: cropped_ARJAN-SINGH_2023-10-21114049_1_png.rf.b69da0570af4e1412f3db4930530b05e.jpg


/usr/local/lib/python3.11/dist-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


Streaming output truncated to the last 5000 lines.
Extracting features from image 4444/9443: 3975470000-jpg_png_jpg.rf.39e54ff36fa9d84b2511649d72549353.jpg
Extracting features from image 4445/9443: 3975470000-jpg_png_jpg.rf.74efec0eebc1fa19b250b1b6d622e1f0.jpg
Extracting features from image 4446/9443: 3975470000-jpg_png_jpg.rf.ead21d51a77dc6fa2ca200eda907fcc1.jpg
Extracting features from image 4447/9443: 3975520000-jpg_png_jpg.rf.6e4fda94da07ac0b60814bb6800c5d21.jpg
Extracting features from image 4448/9443: 3975520000-jpg_png_jpg.rf.8a9facf3c4c69fe5f05f7a6079b6e31d.jpg
Extracting features from image 4449/9443: 3975520000-jpg_png_jpg.rf.ba16dd8d454b141b57f1a939fa0956c2.jpg
Extracting features from image 4450/9443: 3975720000-jpg_png_jpg.rf.33dc4d3cb37a037be1c4a4ec7cc01466.jpg
Extracting features from image 4451/9443: 3975720000-jpg_png_jpg.rf.4e454c9c8614ac7bf9fb2e6be4eacabe.jpg
Extracting features from image 4452/9443: 3975720000-jpg_png_jpg.rf.efaded0394aed816cd0ebe6387521ee2.jpg
Extr

NameError: name 'pd' is not defined

In [ ]:
import pandas as pd

# Check if there is any data
if len(features_list) > 0:
    df_cnn = pd.DataFrame(features_list)
    df_cnn.to_csv(output_csv, index=False)
    print(f"Features saved to: {output_csv}")
else:
    print("No data to save.")


Features saved to: /content/drive/MyDrive/dental_dataset/dental_dataset/cnn_features_train.csv


In [ ]:
df_cnn = pd.read_csv("/content/drive/MyDrive/dental_dataset/dental_dataset/cnn_features_train.csv")
df_manual = pd.read_csv("/content/drive/MyDrive/dental_dataset/dental_dataset/manual_features_train2.csv")

# Merge the data based on image name
df_combined = pd.merge(df_cnn, df_manual, on="image_name", how="inner")
df_combined.to_csv("/content/drive/MyDrive/dental_dataset/combined_features.csv", index=False)

print("Features merged successfully")
print("First 5 rows:")
print(df_combined.head())


Features merged successfully
First 5 rows:
                                          image_name     cnn_0     cnn_1  \
0  cropped_ARJAN-SINGH_2023-10-21114049_1_png.rf....  0.031355  0.619478   
1  cropped_ARMINDER-SINGH_2023-10-21151030_1_png....  0.032178  0.493335   
2  cropped_ARMINDER-SINGH_2023-10-21151030_1_png....  0.034500  0.426229   
3  cropped_ARSHDEEP-SINGH_2023-10-21124811_1_png....  0.042977  0.253108   
4  cropped_ARSHDEEP-SINGH_2023-10-21124811_1_png....  0.057880  0.300909   

      cnn_2     cnn_3     cnn_4     cnn_5     cnn_6     cnn_7     cnn_8  ...  \
0  0.040540  0.084724  0.599837  0.366325  0.540656  0.319573  0.551458  ...   
1  0.094435  0.006230  0.261110  0.587003  0.216620  0.028355  0.780274  ...   
2  0.111969  0.004221  0.245868  0.596356  0.219986  0.033461  0.720438  ...   
3  0.063762  0.197452  0.576374  0.056837  0.271251  0.426348  0.328215  ...   
4  0.074279  0.221280  0.568069  0.045099  0.307710  0.422869  0.331596  ...   

   cnn_2043  cnn_20

In [16]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/dental_dataset/combined_features.csv")
print(df.head())

                                          image_name     cnn_0     cnn_1  \
0  cropped_ARJAN-SINGH_2023-10-21114049_1_png.rf....  0.031355  0.619478   
1  cropped_ARMINDER-SINGH_2023-10-21151030_1_png....  0.032178  0.493335   
2  cropped_ARMINDER-SINGH_2023-10-21151030_1_png....  0.034500  0.426229   
3  cropped_ARSHDEEP-SINGH_2023-10-21124811_1_png....  0.042977  0.253108   
4  cropped_ARSHDEEP-SINGH_2023-10-21124811_1_png....  0.057880  0.300909   

      cnn_2     cnn_3     cnn_4     cnn_5     cnn_6     cnn_7     cnn_8  ...  \
0  0.040540  0.084724  0.599837  0.366325  0.540656  0.319573  0.551458  ...   
1  0.094435  0.006230  0.261110  0.587003  0.216620  0.028355  0.780274  ...   
2  0.111969  0.004221  0.245868  0.596356  0.219986  0.033461  0.720438  ...   
3  0.063762  0.197452  0.576374  0.056837  0.271251  0.426348  0.328215  ...   
4  0.074279  0.221280  0.568069  0.045099  0.307710  0.422869  0.331596  ...   

   cnn_2043  cnn_2044  cnn_2045  cnn_2046  cnn_2047  mean_inte

In [17]:
class DentalCombinedDataset(Dataset):
    def __init__(self, csv_path):
        self.data = pd.read_csv(csv_path)


        self.cnn_columns = [f"cnn_{i}" for i in range(2048)]
        self.manual_columns = [
            "mean_intensity", "std_intensity",
            "dark_pixel_count", "symmetry_score", "image_entropy"
        ]
        self.label_columns = [col for col in self.data.columns if col.startswith("label_")]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]


        cnn_features = row[self.cnn_columns].values.astype(np.float32)


        manual_features = row[self.manual_columns].values.astype(np.float32)

        # Labels
        labels = row[self.label_columns].values.astype(np.float32)

        return (
            torch.tensor(cnn_features),
            torch.tensor(manual_features),
            torch.tensor(labels)
        )

NameError: name 'Dataset' is not defined